# Tipping points

Here we want to look at modelling tipping points. There are thought to be many tipping points in the Earth's climate system. First we will try to model tipping points in the abstract though.

Take this differential equation:
$$
\frac{dx}{dt} = x - x^3 + \lambda = f(x, \lambda)
$$
where $x$ is a dynamic variable and $\lambda$ is a parameter. First let's plot this function $f(x,\lambda)$ on the rhs for a particular value of $\lambda$:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# We'll take lambda=0 for now
l = 1

def f(x, l):
    return x - x**3 + l

ax = plt.figure().add_subplot(111)

x = np.linspace(-1.5, 1.5, 100)
ax.plot(x, f(x, l))

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$f(x, \lambda)$', rotation=0)
ax.hlines(0, -1.5, 1.5, linestyles='dashed')
ax.grid()

plt.show()


## Study the equilibria

The equilibria of the system are when $f(x, \lambda) = 0$ and for $\lambda=0$ there are three equilibria at -1, 0 and 1. When $f(x, \lambda) > 0$ the dynamic variable $x$ moves to the right and when $f(x, \lambda) < 0$ it moves to the left. That means that the middle equilibrium is unstable and the two other equilibria are stable.

A simple way to check this is that if the slope of $f(x,\lambda)$ is positive then the equilibrium is unstable and if the slope is negative then the equilibrium is stable. This corresponds to the sign of $k$ in the linear differential equation:
$$
\frac{dx}{dt} = kx
$$
The slope in this case is $k$ and the equilibrium $x = 0$ is stable if $k < 0$ and unstable if $k > 0$.

## What do the solutions of the differential equation look like?

The solutions of the ODE depend on the initial conditions but will converge to one of the two stable equilibria:

In [ ]:
from scipy.integrate import solve_ivp

# Set lambda = 0
l = 0

def f_rhs(t, x):
    return x - x**3 + l

ax = plt.figure().add_subplot(111)

# Initial values for x. We take a range of values between -1.5 and 1.5
ics = np.linspace(-1.5, 1.5, 10)

t = np.linspace(0, 6, 100)
for ic in ics:
    sol = solve_ivp(f_rhs, (0, 10), [ic], max_step=0.1) #solve initial value problem
    plt.plot(sol.t, sol.y[0])

ax.set_xlabel(r'$t$')
ax.set_ylabel(r'$x(t)$', rotation=0)
ax.grid()

plt.show()


## Changing the parameter

What happens though if we change the parameter $\lambda$?

Try changing $\lambda$ in the plot below. What are the critical values of $\lambda$?

In [ ]:
# Set the value of lambda
l = 0.9

# Since the rhs is polynomial we can solve for real roots exactly with sympy
import sympy as sym
x = sym.Symbol('x')
f_sym = x - x**3 + l
equilibria = [r.n() for r in sym.real_roots(f_sym)]

print('Equilibria:', equilibria)

ax = plt.figure().add_subplot(111)

xv = np.linspace(-1.5, 1.5, 100)
ax.plot(xv, f(xv, l))

ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$f(x, \lambda)$', rotation=0)
ax.hlines(0, -1.5, 1.5, linestyles='dashed')

for eq in equilibria:
    ax.plot(eq, 0, 'ro')

plt.show()

## Bifurcations in the parameter $\lambda$

For some values of $\lambda$ there are 3 equilibria with one being unstable and two being stable. For other values there is only one equilibrium and it is stable. At the boundary between these there are two equilibria which happens at the critical values of $\lambda$. We call these the bifurcation points. If we imagine that our parameter $\lambda$ is itself changing in the time then these critical values of $\lambda$ are the tipping points of the system.

In a simple 1D system like this (a single first order ODE) the conditions for the tipping point are
$$
f(x, \lambda) = 0 \quad \text{and} \quad f_x(x, \lambda) = 0.
$$
In other words the tipping point happens at a value of $x$ and $\lambda$ such that $x$ is an equilibrium ($f(x,\lambda)=0$) and slope of $f$ is zero ($f_x(x,\lambda)=0$). In our case $f(x,\lambda)=x-x^3+\lambda$ so our equations are
$$
x - x^3 + \lambda = 0 \quad \text{and} \quad 1 - 3x^2 = 0
$$
The simple way to solve this is to solve for $x$ in the second equation giving $x = \pm\frac{1}{\sqrt{3}}$ and then substitute each of those solutions into the second equation giving $\lambda = \pm \frac{2\sqrt{3}}{9} = \pm 0.385\cdots$

A way that works more generally for polynomials of any degree (not suitable for hand calculation beyond the quadratic case) is to compute the discriminant of $f(x,\lambda)$ using computer algebra:

In [ ]:
import sympy as sym
x, l = sym.symbols('x, lambda')
f_sym = x - x**3 + l
d = sym.discriminant(f_sym, x)
d

In [ ]:
sym.real_roots(d)

In [ ]:
[r.evalf(5) for r in sym.real_roots(d)]

## Bifurcations as tipping points

We have found the critical values of $\lambda$ as $\pm 0.385\cdots$ and we understand that our system behaves differently either side of the critical values. So far we have considered $\lambda$ to be a constant though.

The idea of a tipping point comes from thinking of $\lambda$ as being something that changes with time as well. We suppose that $\lambda$ is something that changes slowly or that is usually constant but now or in future begin to change. The question then is when $\lambda$ might cross a tipping point.

Why might $\lambda$ change in a real situation? Imagine say that $\lambda$ is the level of CO2 in the atmosphere and it has been constant for a long time. Humans are now pumping out CO2 and that means that $\lambda$ is slowly increasing...

Let's suppose that we start out at the stable equilibrium on the left but $\lambda$ increases slowly. We can suppose that $\lambda = \beta t$ where $\beta$ is positive but small. What happens then?

In [ ]:
# Animate what happens when lambda changes

import matplotlib.animation as animation

# Rate of increase of lambda
beta = +0.1

# Initial value of x
x0 = -1.0

def f(t, x):
    l = beta * t
    return x - x**3 + l

fig = plt.figure()
ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

delta_t = 0.1

sol = solve_ivp(f, (0, 20), [x0], max_step=delta_t)

xvals = np.linspace(-1.5, 1.5, 100)

def animate(i):
    ax1.clear()
    ax2.clear()
    ax1.plot(xvals, f(delta_t * i, xvals))
    ax1.set_xlabel(r'$x$')
    ax1.set_ylabel(r'$f(x, \lambda)$', rotation=0)
    ax1.grid()
    ax1.plot(sol.y[0][i], 0, 'o')
    ax2.plot(sol.t[:i], sol.y[0][:i])
    ax2.set_ylim([-1.5, 1.5])
    ax2.set_xlabel(r'$t$')
    ax2.set_ylabel(r'$x(t)$')
    ax2.set_xlim([0, 20])

ani = animation.FuncAnimation(fig, animate, frames=200, interval=100)
from IPython.display import HTML
HTML(ani.to_jshtml())

Have a look at the simulator here for this:

[https://oscarbenjamin.github.io/tippingpoints/](https://oscarbenjamin.github.io/tippingpoints/)



In [ ]:
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import numpy as np

P = 4
beta = 5 # 6.25
gamma = 1
K = 1

def beta(t):
    return 5 + 0.01*t

def ice(t, h):
    dhdt = P - beta(t)/(1 + h) - gamma*h
    if h <= 0 and dhdt < 0:
        dhdt = 0
    return dhdt

# h = np.linspace(0, 4, 100)
# ax = plt.figure().add_subplot(111)
# ax.plot(h, ice(0, h))
# ax.grid()

# plt.show()

t = np.linspace(0, 200, 100)
sol1 = solve_ivp(ice, [0, 200], [0.3], t_eval=t)
sol2 = solve_ivp(ice, [0, 200], [2.6], t_eval=t)
plt.plot(sol1.t, sol1.y[0])
plt.plot(sol2.t, sol2.y[0], label=r'$h(t)$')
plt.plot(sol1.t, beta(sol1.t), label=r'$\beta(t)$')
plt.hlines(6.25, 0, 200, 'r', '--')
plt.vlines(125, 0, 7, 'r', '--')
plt.grid()
plt.legend()
plt.show()
